#**Neural Network for Human Activity Recognition**

## Introduction
This notebook demonstrates a complete TinyML workflow for HAR on the Nicla Vision.
We will approach this in two parts:

* **Part 1: Binary Classification:** Differentiating between 'Sitting' (Idle) and 'Flick' (Action).
* **Part 2: Multi-class Classification:** Detecting 'Sitting', 'Flick', and 'Up-Down' gestures.

## Workflow
1.  **Feature Engineering:** Apply a sliding window to raw IMU data and extract statistical features (Mean & Std Dev).
2.  **Model Training:** Train Scikit-Learn MLP models (1 Hidden Layer) for both scenarios.
3.  **Visualization:** Visualize the Neural Network architecture using Graphviz.
4.  **Deployment:** Export the trained weights of the Multi-class model to `mlp_params.py` for the edge device.
"""

# IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from graphviz import Digraph
import json

# Setup plotting
sns.set(style='darkgrid')

: 


# 1. DATA LOADING & PREPARATION


In [ ]:
# Ensure these CSV files are uploaded to your environment
cols = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

# Load Datasets
try:
    df_sit = pd.read_csv('sit_20251228_032604.csv')[cols]
    df_flick = pd.read_csv('rlswing_20251228_032948 (1).csv')[cols]
    df_updown = pd.read_csv('updown_20251228_033225.csv')[cols]

    # Assign Labels
    df_sit['label'] = 0      # Sitting
    df_flick['label'] = 1    # Flick
    df_updown['label'] = 2   # Up-Down

    print(f"Loaded: Sit({len(df_sit)}), Flick({len(df_flick)}), UpDown({len(df_updown)})")
except Exception as e:
    print(f"Error loading data: {e}. Please ensure CSV files are present.")

# 2. FEATURE ENGINEERING

In [ ]:
def extract_features(window):
    """
    Computes statistical features for a single window.
    Input: DataFrame window (N rows x 6 columns)
    Output: 1D Array [Mean_ax, ..., Mean_gz, Std_ax, ..., Std_gz] (12 features)
    """
    means = window.mean().values
    stds = window.std().values
    return np.hstack([means, stds])

def create_dataset(dfs, window_size=20, step_size=1):
    """
    Slides a window over the list of dataframes to create X (features) and y (labels).
    """
    X = []
    y = []

    for df in dfs:
        label = df['label'].iloc[0]
        # Drop label column for feature extraction
        data = df.drop(columns=['label'])

        for i in range(0, len(data) - window_size, step_size):
            window = data.iloc[i:i+window_size]
            features = extract_features(window)
            X.append(features)
            y.append(label)

    return np.array(X), np.array(y)

In [ ]:
df_sit.head(20)

In [ ]:
df_sit.head(20).describe()

In [ ]:
df_tem = create_dataset([df_sit, df_flick], window_size=20)
df_tem

# 1. Unpack the tuple (Features, Labels)
X_data, y_data = df_tem

# 2. Define Column Names (Mean + Std for 6 axes)
cols = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
feature_cols = [f"Mean_{c}" for c in cols] + [f"Std_{c}" for c in cols]

# 3. Create DataFrame from Features
df_features = pd.DataFrame(X_data, columns=feature_cols)

# 4. Add Label Column
df_features['label'] = y_data

# 5. Display the Table
print(f"Total Samples: {len(df_features)}")
df_features.head(1)

# 3. PART 1: BINARY CLASSIFICATION (Sit vs. Flick)

In [ ]:
print("\n--- PART 1: BINARY CLASSIFICATION ---")

# Create Binary Dataset
X_bin, y_bin = create_dataset([df_sit, df_flick], window_size=20)
print(f"Binary Dataset Shape: {X_bin.shape}")

# Split & Scale
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_bin, y_bin, test_size=0.2, random_state=42)
scaler_bin = StandardScaler()
X_train_b_scaled = scaler_bin.fit_transform(X_train_b)
X_test_b_scaled = scaler_bin.transform(X_test_b)

# Train Binary MLP (1 Hidden Layer)
clf_bin = MLPClassifier(hidden_layer_sizes=(8,), activation='relu', max_iter=1000, random_state=42)
clf_bin.fit(X_train_b_scaled, y_train_b)

# Evaluate
print(f"Binary Test Accuracy: {clf_bin.score(X_test_b_scaled, y_test_b):.2f}")
print(classification_report(y_test_b, clf_bin.predict(X_test_b_scaled), target_names=['Sit', 'Flick']))

# 4. PART 2: MULTI-CLASS CLASSIFICATION (Sit vs. Flick vs. UpDown)

In [ ]:

print("\n--- PART 2: MULTI-CLASS CLASSIFICATION ---")

# Create Multi-class Dataset
X_multi, y_multi = create_dataset([df_sit, df_flick, df_updown], window_size=20)
print(f"Multi-class Dataset Shape: {X_multi.shape}")

# Split & Scale
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)
scaler_multi = StandardScaler()
X_train_m_scaled = scaler_multi.fit_transform(X_train_m)
X_test_m_scaled = scaler_multi.transform(X_test_m)

# Train Multi-class MLP (1 Hidden Layer, slightly larger)
# 12 Inputs -> 16 Hidden Neurons -> 3 Outputs
clf_multi = MLPClassifier(hidden_layer_sizes=(16,), activation='relu', max_iter=1000, random_state=42)
clf_multi.fit(X_train_m_scaled, y_train_m)

# Evaluate
print(f"Multi-class Test Accuracy: {clf_multi.score(X_test_m_scaled, y_test_m):.2f}")
print(classification_report(y_test_m, clf_multi.predict(X_test_m_scaled), target_names=['Sit', 'Flick', 'UpDown']))

In [ ]:
for i in clf_multi.coefs_:
  print(i.shape)

**5.Model Summary**

In [ ]:


def print_model_summary(layer_sizes):
    """
    Prints a Keras-style model summary for a simple MLP.
    """
    print("-" * 65)
    print(f"{'Layer (type)':<20} {'Output Shape':<20} {'Param #':<20}")
    print("=" * 65)

    total_params = 0

    for i in range(len(layer_sizes) - 1):
        n_in = layer_sizes[i]
        n_out = layer_sizes[i+1]

        # Calculate Weights + Biases
        weights = n_in * n_out
        biases = n_out
        params = weights + biases
        total_params += params

        layer_type = "Input Layer" if i == 0 else "Hidden Layer"
        if i == len(layer_sizes) - 2: layer_type = "Output Layer"

        print(f"{layer_type:<20} (None, {n_out:<12}) {params:<20}")
        print("-" * 65)

    print(f"Total params: {total_params}")
    print(f"Trainable params: {total_params}")
    print(f"Non-trainable params: 0")
    print("=" * 65)



# Define your architecture
# 12 Input Features (Mean/Std of 6 axes) -> 16 Hidden Neurons -> 3 Output Classes
arch = [12, 16, 3]

# 1. Print Summary
print_model_summary(arch)



## 6. VISUALIZATION

In [ ]:
from graphviz import Digraph

def draw_mlp_graphviz(layer_sizes):
    dot = Digraph(format="png")
    dot.attr(rankdir="LR")

    # 1. INCREASE SPACE: 'ranksep' controls space between layers (default is ~0.5)
    dot.attr(ranksep='2.5')
    # Optional: 'nodesep' controls space between neurons in the same layer
    dot.attr(nodesep='0.3')

    for i, size in enumerate(layer_sizes):
        with dot.subgraph(name=f"cluster_{i}") as c:
            c.attr(label=f"Layer {i} ({size} neurons)")

            # Distinguish Input, Hidden, Output colors
            color = "lightblue" if i == 0 else "lightgrey"
            if i == len(layer_sizes)-1: color = "lightgreen"

            c.attr(style='filled', color=color)
            for j in range(size):
                c.node(f"L{i}N{j}", label=f"N{j}", shape="circle", style="filled", fillcolor="white")

    for i in range(len(layer_sizes) - 1):
        for j in range(layer_sizes[i]):
            for k in range(layer_sizes[i + 1]):
                # Only draw a few edges to keep it clean if the network is large
                if layer_sizes[i] > 16 and j > 2 and j < layer_sizes[i]-3: continue

                # 2. GREY LINES: Set color='gray' (or a hex code like '#C0C0C0')
                dot.edge(f"L{i}N{j}", f"L{i+1}N{k}", color='gray')

    return dot

# Define your architecture
# 12 Input Features -> 16 Hidden Neurons -> 3 Output Classes
arch = [12, 16, 3]

# Visualize
draw_mlp_graphviz(arch)

7. EXPORT WEIGHTS (For deployment)

In [ ]:
# We export the Multi-class model weights to be used in main.py
print("\n--- EXPORTING WEIGHTS ---")

W1, W2 = clf_multi.coefs_
b1, b2 = clf_multi.intercepts_

norm_params = {
    "mean": scaler_multi.mean_.tolist(),
    "scale": scaler_multi.scale_.tolist()
}
nn_params = {
    "W1": W1.round(4).tolist(),
    "b1": b1.round(4).tolist(),
    "W2": W2.round(4).tolist(),
    "b2": b2.round(4).tolist(),
}

with open("mlp_params.py", "w") as f:
    f.write("# Auto-generated from MLP_for_HAR.ipynb\n")
    f.write(f"norm_params = {json.dumps(norm_params, indent=4, )}")
    f.write("\n")
    f.write(f"nn_params = {json.dumps(nn_params, indent=4, )}")


print("Done! 'mlp_params.py' saved. Upload this file to your Nicla Vision.")

In [ ]:
!cat mlp_params.py